# MT3510 Group Project Report


## **Introduction**
This report will explore **Deterministic Finite Automata (DFA)**, an important concept in theoretical computer science and mathematics. A DFA is a finite state machine that takes in string of symbols and accepts or rejects the string based on the sequence of states. DFAs have many practical applications, including recognising formal languages, malware detection, text processing, user interface and compilers. The theoretical foundations were laid by the work of Alan Turing in the 1930s, creating the Turing machine, which helped in the development of finite state machines that became essential for early computer engineers.

We will specifically explore how to identify a **synchronising DFA**, a special type of DFA that contains a reset word in its alphabet. The Cerny Conjecture states that $(n-1)^2$ is the largest length of a reset word for a DFA with n states. The conjecture made by Jan Cerny in 1969 still remains an unsolved problem in Computer science. We will discover that synchronizing DFAs correspond to a transformation semigroup, a set of functions that is closed under composition and contains a constant map (if synchronizing). One of the methods used involves creating a pair digraph for a DFA to identify if the given DFA is sychronizing. This involves finding a transformation that sends any pair of states to the same state.

We will also be exploring **isomorphisms** between two DFAs that have various levels of strictness, determining if synchronizing is an invariant property. Specifically, we will analyse strict, weak and semi isomorphisms, which allows us to compare DFAs and evaluate their properties.

## 2. Description of Implementation


### 2.1 Strict Isomorphism Implementation

We aim to find if two given DFAs are strictly isomorphic. To do this, we will test for if there exists an isomorphism between the DFAs.

**The general idea:**

- Define a function is_strict_isomorphism to test if two DFAs $A_1$ and $A_2$ are strictly isomorphic

*To be strictly isomorphic*

- Need a bijection from $Q_1$ to $Q_2$, so must have same cardinalities ($|Q_1|=|Q_2|$), return False if not

- Need $f(F_1)=F_2$, since $f$ is bijection, must have $|F_1|=|F_2|$, return False if not

Now we can construct the mapping $f$ using backtrack search

**Note:** that we assumed that $Q={0,1,...,n-1}$ for some natural number $n$, with $|Q_1|=|Q_2|$ if $f$ is an injection, it is a bijection

- Create a list to track the used Q2 states, this insures injectivity of $f$

- Want $f$ to map $(q_s)_1$ to $(q_s)_2$, set $f((q_s)_1)=(q_s)_2$, mark $(q_s)_2)$ as used

- Define the recursive backtracking function, $dive(v)$, on a vertex $v$

- $dive(v)$ does:

  - check if every state is mapped, if that happens, verify isomorphism conditions on $f$, returning True if passes all

  - if not, then if $v$ already mapped then move to next, until an unmapped vertex is found

  - using a loop, try every *unused* state $x$ of $Q_2$ as image of $v$ under $f$, verifing the isomorphism conditions hold, if violated, break out and try the next $x$ 

  - do these two lines: 1. 'if $dive(v+1)$: return True', and after that 2. revert the changes i.e. $f[v]$ to $-1$ and set "usedness" of $x$ back to unusued (this line will only be done if False is returned)

  - these two lines are crucial, since $dive(v+1)$ will eventually reach line 1, which procedes with $dive(v+2)$ and so on. Finally $dive(n)$ will be reached, this is the 'every(last) vertex is mapped' case, now dive verifies the isomorphism conditions and either return a True or a False

  - if True is returned we found a solution, so propagate the True upwards so that $dive(v)$ returns True

  - if False is returned, this is a dead end, all possible constructions from $f[v]=x$ has failed, hence does line 2 (which can be interpreted as going one step back), and explores other possiblities. This might succeed or have to go further back.

  - lastly $dive(v)$ returns False. This will only be triggered if all possible constructions from $v$ has failed, hence $v$ is a dead end, so need to move back, to $v-1$.

Now out of dive

- let is_strict_isomorphism return $dive(0)$

- this will either fully construct $f$ and return a True, or will exhaust all possible constuctions from $0$ and finally having $dive(0)$ returning a False

**Discussion about exsistance of a bijection**

Since being strictly isomorphic means 'there exists an isomorphism between the DFAs' and we constucted a specific one. If the $f$ constructed fails to be an isomorphism, could there still exist an isomorphism $g$, making the DFAs isomorphic? 

To explain this, note that backtracking search exhausts every possible bijection, so is_strict_isomorphism searches *every* bijection that maps $(q_s)_1$ to $(q_s)_2$. It tries every possible mapping to an unmapped state, with this, if an isomorphism existis, it *will* find one. So returning a False really means there does not exist such an isomorphism.


### 2.2 Semi-Isomorphism Implementation
Aim: Given two DFAs $A_1$, $A_2$, determine whether they are semi-isomorphic and, if so, return the bijections $f: Q_1 → Q_2$ and $g: Σ_1 → Σ_2$.

Method: The algorithm works in the following manner:

### 1. Root Identification: 

We are given that there is a 'root' state from which all other states are reachable. 

 - Convert the two DFAs into directed graphs using the 'networkx' package.
 - Compute the Strongly Connected Components (SCCs) of each graph, strongly connected components are components from which all other states within the component are reachable from each other.
 - Construct the condensation graph, where each SCC is collapsed into a single node, $C$ is the set of SCCs.
 - Identify the SCC with out-degree of value $|C| - 1$, meaning it can reach all other SCCs. All states in this SCC are root candidates.

### 2. Backtrack Algorithm:

Choose a candidate pair of roots from the chosen SCC:
- $\text{root}_1 \in Q_1$ (from $A_1$)
- $\text{root}_2 \in Q_2$ (from $A_2$)
  
We set the mapping $f($root1$) = $ root2 for the candidate pair of roots. The algorithm then attempts to extend this into full bijections $f$ and $g$ using a backtracking search.


### 3. Backtracking Algorithm

The backtracking algorithm incrementally constructs the bijections $f$ and $g$ while ensuring consistency with the transition functions.

- For each mapped pair of states $(q_1, q_2)$:
  - For each symbol $a \in \Sigma_1$:
    - Try to assign a symbol $b \in \Sigma_2$ such that:
      $$
      f(\delta_1(q_1, a)) = \delta_2(q_2, b)
      $$

- During this process, if $a$ is already mapped by $g$, reuse the mapping, otherwise, try all unused symbols in $\Sigma_2$.

- If a conflict occurs, the algorithm 'backtracks' (returns) to the previous step and tries to a different mapping,

- The algorithm continues until complete bijections for both $f$ and $g$ are found, or all possible root pairings and mappings have been exhausted and the no semi-ismorphisms exist.


## 3. Code


### 3.1 Strict Isomorphism


In [ ]:
def is_strict_isomorphism(A1,A2):
    """
    Tests if two DFAs A1=(Q1,Σ,τ1,qs1,F1) and A2=(Q2,Σ,τ2,qs2,F2) are strictly isomorphic.
    Assumes they have the common alphabet Σ.
    Assumes every state in Q is reachable from qs.
    Uses backtrack search.

    Parameters:
    - A1, A2: Two DFAs
    Arguments of A1, A2:
    - Q1, Q2: some finite set of states
    - Σ: some finite alphabet (a set)
    - τ1, τ2: transition function τi:Qi×Σ→Qi for i=1,2 (a dict)
    - qs1,qs2: start state
    - F1, F2: set of accepting or final states

    Returns: 
    Boolean True or False
    """
    # Store the arguments of the DFAs in variables(if not already in this form)
    Q1,Σ,τ1,qs1,F1 = A1
    Q2,Σ,τ2,qs2,F2 = A2
    # Test the cardinalities of the Qs and the Fs are equal
    if len(Q1)!=len(Q2):
        return False
    if len(F1) != len(F2):
        return False
    # Build the mapping, since Q1 = Q2 = {0,1,...,n-1}, injective implies bijective.
    n = len(Q1)
    f = [-1]*n   # for q in Q1, f[q]=f(q), the image of q in Q2, -1 means not mapped yet.
    used = [False] * n   # Track which states of Q2 are already used, ensures f injective, hence bijective
    # Map qs1 to qs2 i.e. f(qs1)=qs2
    f[qs1] = qs2
    used[qs2] = True
    # Define the recursive backtracking function
    def dive(v):   # v is a vertex/state, will run dive from v=0
        if v == n:   # If we have mapped all vertices, verify the conditions are satisfied
            for q in Q1:
                if (q in F1)!=(f[q] in F2):
                    return False
                for a in Σ:
                    if τ2[(f[q], a)] != f[τ1[(q, a)]]:
                        return False
            return True   # f satisfies the conditions and is a bijection, i.e. isomorphism, so A1 A2 stricly isomorphic.
        if f[v] != -1:
            return dive(v+1)     # If the current vertex is already mapped, skip to the next
        # Try every unused state in Q2, x, as a image for v (i.e. to see if we can set f(v)=x)
        for x in Q2:
            if used[x]:   
                continue   # x has been used, get a new x
            # Check f(F1)=F2, if false can jump right into next loop, saving time 
            if (v in F1) != (x in F2):
                continue
            # Check consistency with already mapped vertices u in Q1
            consistent = True
            for u in Q1:
                if f[u] == -1:
                    continue   # Find those mapped vertices, skipping the unmapped
                for a in Σ:
                    if τ1[(u,a)] == v:   # If from u we go to v on some symbol,
                        if τ2[(f[u],a)] != x:
                            consistent = False   # x is not f(v), since f(v)=f(τ1(u,a))=τ2(f(u),a)
                            break
                    if τ1[(v,a)] == u:   # If from v we go to u on some symbol,
                        if τ2[(x,a)] != f[u]:   
                            consistent = False   # if x=f(v), then f(u)=f(τ1(v,a))=τ2(f(v),a)=τ2(x,a)
                            break
                if not consistent:   # If consistent is False, break out immediately, if True, loop on with next u.
                    break
            if not consistent:   # If consistent is False, this x is inconsistent, go to next x
                continue
            # If reaches here, consistent is True throughout, so try to map v to x, then run dive reapeatedly.
            f[v] = x
            used[x] = True
            if dive(v+1):
                return True   # To reach here, must reached v=n and f[v:n] satisfies the isomorphism conditions.
            f[v]=-1   # Reaching here implies that something went wrong in the if dive(v+1), dive(v+2),...loops
            used[x] = False    # so this x does not work try with another x and all the succeeding vs and xs are reverted
        # Reaching here means that no x worked, so v cant have a image satisfying the isomorphism conditions.
        return False   # This will show that v is a 'dead end' after every possible extenstion, so having to go to higher up (v-1)
    return dive(0)   # This will either fully construct f and returning a True, or,  after trying every possible path, return to dive(0) and return a False

### 3.2 Semi-Isomorphism


In [ ]:
import networkx as nx

def is_semi_isomorphism(A1, A2):
    """
    Tests if two DFAs A1=(Q1,Σ1,τ1,qs1,F1) and A2=(Q2,Σ2,τ2,qs2,F2) are semi-isomorphic.

    Semi-isomorphism requires bijections f: Q1 → Q2 and g: Σ1 → Σ2 such that:
        τ2(f(q), g(a)) = f(τ1(q, a))  for all q ∈ Q1, a ∈ Σ1

    Start states and final states are ignored.
    Assumes that in each DFA there is at least one state that can reach all other states.

    Returns: (True, f, g)  or  (False, None, None)
    """
    Q1, sigma1, tau1, qs1, F1 = A1
    Q2, sigma2, tau2, qs2, F2 = A2

    if len(Q1) != len(Q2) or len(sigma1) != len(sigma2):
        return False, None, None

    # Build directed multigraphs for SCC analysis
    G1 = nx.MultiDiGraph()
    for q in Q1:
        for a in sigma1:
            G1.add_edge(q, tau1[(q, a)], label=a)

    G2 = nx.MultiDiGraph()
    for q in Q2:
        for a in sigma2:
            G2.add_edge(q, tau2[(q, a)], label=a)

    sccs1 = list(nx.strongly_connected_components(G1))
    sccs2 = list(nx.strongly_connected_components(G2))
    C1 = nx.condensation(G1, scc=sccs1)
    C2 = nx.condensation(G2, scc=sccs2)

    # Find states that can reach all others (root candidates)
    reaching_states1 = []
    reaching_states2 = []

    for node in C1.nodes():
        if len(nx.descendants(C1, node)) == len(C1.nodes()) - 1:
            reaching_states1.extend(sccs1[node])

    for node in C2.nodes():
        if len(nx.descendants(C2, node)) == len(C2.nodes()) - 1:
            reaching_states2.extend(sccs2[node])

    for root1 in reaching_states1:
        for root2 in reaching_states2:
            f = {root1: root2}
            queue = [(root1, a) for a in sigma1]
            result = _backtrack_semi(f, {}, queue, tau1, tau2, sigma1, sigma2, Q1)
            if result is not None:
                return True, result[0], result[1]

    return False, None, None


def _backtrack_semi(f, g, queue, tau1, tau2, sigma1, sigma2, Q1):
    """Recursive backtracking helper for is_semi_isomorphism."""
    if not queue:
        if (len(f) == len(Q1) and len(g) == len(sigma1)
                and len(set(f.values())) == len(Q1)
                and len(set(g.values())) == len(sigma1)):
            return f, g
        return None

    q, a = queue[0]
    rest = queue[1:]
    fq = f[q]
    q_next = tau1[(q, a)]

    if a in g and q_next in f:
        # Case 1: both g(a) and f(q_next) already known — just verify
        if tau2[(fq, g[a])] != f[q_next]:
            return None
        return _backtrack_semi(f, g, rest, tau1, tau2, sigma1, sigma2, Q1)

    elif a in g and q_next not in f:
        # Case 2: g(a) known, f(q_next) unknown — determine f(q_next) from τ2
        target = tau2[(fq, g[a])]
        if target in f.values():
            return None
        new_f = {**f, q_next: target}
        return _backtrack_semi(new_f, g, rest + [(q_next, b) for b in sigma1],
                               tau1, tau2, sigma1, sigma2, Q1)

    elif a not in g and q_next in f:
        # Case 3: f(q_next) known, g(a) unknown — try all unused symbols
        target = f[q_next]
        for b in sigma2:
            if b not in g.values() and tau2[(fq, b)] == target:
                result = _backtrack_semi(f, {**g, a: b}, rest,
                                         tau1, tau2, sigma1, sigma2, Q1)
                if result is not None:
                    return result
        return None

    else:
        # Case 4: neither known — try all unused (b, target) pairs
        for b in sigma2:
            target = tau2[(fq, b)]
            if b not in g.values() and target not in f.values():
                new_f = {**f, q_next: target}
                new_g = {**g, a: b}
                new_queue = rest + [(q_next, b2) for b2 in sigma1]
                result = _backtrack_semi(new_f, new_g, new_queue,
                                         tau1, tau2, sigma1, sigma2, Q1)
                if result is not None:
                    return result
        return None


## Effect of Isomorphism Type on Synchronisation

### Strict isomorphism
Strict isomorphism will fully preserve synchronization properties of DFAs (if at all). This is since two strictly isomorphic DFAs can be thought as renaming the states, with structures identical, so if a reset word exists in one, it also exists in the other, but probably  resets to a different state.

To explicitly show this, let $A_1=(Q_1,Σ,τ_1,(q_s)_1,F_1)$ and $A_2=(Q_2,Σ,τ_2,(q_s)_2,F_2)$ be strictly isomorphic, $f$ be an isomorphism between them. For $x∈Q_1$ define $t_1(x,w)=y$ to be "transition function of words", i.e using a word $w$ on a state $x$ gives state $y$. With $w=w_1w_2...w_n$ where each $w_i∈Σ$ for finite natural $i$, $t_1(x,w)$ is basically $τ_1(...(τ_1(τ_1(x,w_1),w_2),...,),w_n)$. Using induction on length of $w$, $t_2(f(q),w)=f(t_1(q,w))$ for all $q∈Q_1$ and all word $w$ over $Σ$ can be shown to hold, but too lengthy to write here.

If $A_1$ is synchronizing, there exists a reset word $w$ over $Σ$, and resets to a state $z∈Q_1$, i.e. $t_1(q,w)=z$ for all $q∈Q_1$. Then $f(z)=f(t_1(q,w))=t_2(f(q),w)$ for all $f(q)=q_2∈Q_2$ (by surjectivity of $f$), so $w$ is a reset word and $f(z)∈Q_2$ is the state that is resetted to, so $A_2$ is synchronizing.


### Semi-isomorphism 
Since a semi-isomoprhism is defined by the two bijections $f: Q_1 → Q_2$ and $g: Σ_1 → Σ_2$, the defined mapping between the two DFAs conserves a synchronising word, for example, if a synchronising word $w'$ exists for $A_2$, the inverse mapping of $g^-1$ identifies the corresponding word $w$ in $A_1$.

This follows from:

* Structural Symmetry:
Since $f$ and $g$ are bijections, the two DFAs have identical structure. Therefore, if a sequence of transitions in $A_1$ leads all states to a single state, the corresponding sequence in $A_2$ must do the same.

* Edge Preservation:
Because transitions, edges and their labels are preserved under the bijections, the reset behaviour of the word is maintained. This guarantees that synchronising words are preserved.

* Bijective Property:
The converse also holds; if $A_1$ has a synchronising word, then $A_2$ has one and vice-versa, if $A_1$ does not have a synchronising word, then neither does $A_2$.

In conclusion, since semi-isomorphism preserves transitions, their origins, and destinations, the synchronisation behavior of the DFA is fully preserved.



## 4. Demonstration


### 4.1 Strict Isomorphism — Examples


In [ ]:
# Strict isomorphism demonstration examples go here


### 4.2 Semi-Isomorphism — Examples

Four examples are shown below. We report whether the DFAs are semi-isomorphic and why the result is expected.


In [1]:
def report_semi(A1, A2, label=""):
    ok, f, g = is_semi_isomorphism(A1, A2)
    result = "SEMI-ISOMORPHIC" if ok else "NOT semi-isomorphic"
    print(f"{label}  is  {result}")
    if ok:
        print(f"   f = {f}")
        print(f"   g = {g}")
    print()

# Example 1: Renamed states 
# A2 is A1 with states {0,1} renamed to {10,11}. Expect: semi-isomorphic.
A1_ex1 = ({0,1}, {0,1}, {(0,0):1,(0,1):0,(1,0):1,(1,1):0}, 0, {0})
A2_ex1 = ({10,11}, {0,1}, {(10,0):11,(10,1):10,(11,0):11,(11,1):10}, 10, {10})
print(report_semi(A1_ex1, A2_ex1, "Example 1 (renamed states)"))

# Example 2: Self-loop mismatch 
# A1 has self-loops on symbol 0; A2 has none. Expect: NOT semi-isomorphic.
A1_ex2 = ({0,1}, {0,1}, {(0,0):0,(0,1):1,(1,0):1,(1,1):0}, 0, set())
A2_ex2 = ({10,11}, {0,1}, {(10,0):11,(10,1):11,(11,0):10,(11,1):10}, 10, set())
print(report_semi(A1_ex2, A2_ex2, "Example 2 (self-loop mismatch)"))

# Example 3: Different start/accepting states
# Identical transition structure, different start and accepting states.
# Semi-isomorphism ignores both. Expect: semi-isomorphic.
A1_ex3 = ({0,1,2},{0,1},{(0,0):0,(0,1):1,(1,0):1,(1,1):2,(2,0):2,(2,1):0},0,{0})
A2_ex3 = ({10,11,12},{0,1},{(10,0):10,(10,1):11,(11,0):11,(11,1):12,(12,0):12,(12,1):10},10,{10})
print(report_semi(A1_ex3, A2_ex3, "Example 3 (different start/accepting states)"))

# Example 4: Swapped alphabet roles
# In A1 symbol 0 gives self-loops; in A2 symbol 1 gives self-loops.
# Expect: semi-isomorphic (non-trivial g).
A1_ex4 = ({0,1,2},{0,1},{(0,0):0,(0,1):1,(1,0):1,(1,1):2,(2,0):2,(2,1):1},0,{0})
A2_ex4 = ({10,11,12},{0,1},{(10,0):11,(10,1):10,(11,0):12,(11,1):11,(12,0):11,(12,1):12},12,{11})
print(report_semi(A1_ex4, A2_ex4, "Example 4 (swapped alphabet roles)"))


NameError: name 'is_semi_isomorphism' is not defined

## 5. Description of Results


### 5.1 Strict Isomorphism

*(Results and description to be completed by the team.)*


### 5.2 Semi-Isomorphism

* Semi-Isomorphism Existence Cases (Examples 1, 3, 4)
For these examples, the algorithm succesfully finds bijections f and g satisfying τ2(f(q), g(a)) = f(τ1(q, a))   for all states q and symbols a.

- In Example 1 of Renamed States: There are two states in each DFA, and the state mapping follows f: 0 → 10, 1 → 11, and the alphabet mapping is the g: identity (unchanged), hence, we see a semi-isomorphism.

- In Example 3 of Altered Start and Accepting states: again we see that this condition does not influence a semi-isomorphism, as it is not a condition, the same is not true for weak and strong isomorphisms. Again we see the mapping g: identity and the the mapping for f: 0 → 10, 1 → 11, 2 → 12, where the states form a 3-cycle, hence there exists a semi-isomorphism.

- In Example 4: Asymmetric Structure with Symbols Swapped: this DFA has three states with an asymmetric structure, even though the alphabet is the same, the roles of the symbols are swapped between the DFAs. In A1, symbol 0 produces self-loops, however, in A2, symbol 1 produces self-loops. The asymmetry ensures a unique mapping for f and demonstrates a non-trivial semi-isomorphism.

* Semi-Isomorphism Non-Existence Case (Examples 2)

- In Example 2: Self-Loop Non-Existence: in this example, A1 has self-loops constructed by 0, while A2 has no self-loops at all. There exists no possible mapping of states that can satisfy this, hence, no semi-isomorphism exists.


## 6. Analysis of Results


### 6.1 Strict Isomorphism

*(Analysis to be completed by the team.)*


### 6.2 Semi-Isomorphism

### Were the results as expected?

Yes, all four outcomes match the predictions in the example descriptions, and the returned bijections can be verified.

**Positive cases (Examples 1, 3, 4)**

Examples 1 and 3 are straightforward: the two DFAs are structurally identical with renamed states, so $g$ is the identity on the alphabet and $f$ is a simple offset $f(i) = 10 + i$.

For Example 4, this isn't the case The backtracking search starts at root pair $(0, 10)$ and immediately encounters symbol $0$. In $A_1$, $(0, 0) \to 0$ is a self-loop; the algorithm must find $b \in \{0,1\}$ such that $\tau_2(10, b) = 10$. Checking: $\tau_2(10, 0) = 11 \neq 10$, but $\tau_2(10, 1) = 10$. So the only consistent assignment is $g(0) = 1$, and all remaining constraints follow easily with no further backtracking needed. This shows that the algorithm recovers a non-trivial $g$ efficiently from a single forced step.


**Negative case (Example 2)**

Failure is detected at the first undetermined step (Case 4 of `backtrack`). For any root pairing, the algorithm needs $\tau_2(f(q), b) = f(q)$ to match $A_1$'s self-loop, but no state in $A_2$ has a self-loop under any symbol. Every candidate $b$ is immediately rejected, for every root pairing.

## Factors influencing the algorithm

1. **Alphabet asymmetry** - The symbol mapping isn’t always identity; a single root constraint often fixes it automatically.  
2. **State role asymmetry** - States with different roles (cycle vs. tail) immediately determine the mapping, speeding up the search.  
3. **Self-loop check** - Fails instantly if required self-loops aren’t present.  
4. **Root SCC size** - Bigger initial SCCs mean more root pairings, so slower processing.



## 7. Who Did What

We divided the project into two parts: Finlay, James, and Somhairle worked on Part 1 (synchronisation); Elena, Ella, and Fei worked on Part 2 (isomorphism). Within Part 2, Fei implemented strict isomorphism, Elena implemented semi-isomorphism, and Ella implemented weak isomorphism. Each team member wrote the description and analysis for their own component; the group then collaborated on the introduction, Part 3 extensions, and final report assembly.


## 8. Beyond the Project Specification

*(To be completed — describe any Part 3 contributions here.)*


## 9. Use of AI

The original recursive Backtrack implementation was converted to an iterative stack-based approach to avoid Python recursion limits. Claude suggested the stack approach.
